#### SimpleNN Training

This notebook trains SimpleNN models (simple feedforward neural networks) for quantum error correction.

**Purpose:** Baseline comparison against GNN-based decoders to determine if graph structure helps.

**Key Differences from GNN training:**
- Input: Flat syndrome arrays `[batch, num_detectors]`
- No graph structure - just raw syndrome measurements
- Uses `FlatDatasetCache` instead of `DatasetCache`

For hyperparameter tuning, see `code/nn/tuning.ipynb`.

#### Generate Flat Dataset Cache

Generate flat syndrome array datasets for SimpleNN training.
This only needs to be run once per distance.

In [1]:
import sys
from pathlib import Path

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('..')

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
from tqdm import tqdm
import gc

# Import from benchmark_models.py
from benchmark_models import (
    SurfaceCodeSampler,
    SimpleNNModel,
    SimpleNN,
    FlatDatasetCache,
    count_logical_errors,
)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
Using device: cuda
GPU: NVIDIA GeForce GTX 1080


In [2]:
# ============================================================
# CONFIGURATION: Define which flat datasets to generate
# ============================================================

# Standard error rates (same as graph datasets for fair comparison)
STANDARD_P_VALUES = [0.001, 0.003, 0.005, 0.007]
STANDARD_P_WEIGHTS = [0.25, 0.25, 0.25, 0.25]

# Define datasets to generate
# Each entry: (name, distance, n_samples)
FLAT_DATASETS_TO_GENERATE = [
    ("d3_baseline", 3, 1_000_000),
    ("d5_baseline", 5, 1_000_000),
    ("d7_baseline", 7, 1_000_000),
    ("d9_baseline", 9, 1_000_000),
    # ("d11_baseline", 11, 1_000_000),
    # ("d13_baseline", 13, 1_000_000),
]

print("Flat datasets configured for generation:")
print(f"  Error rates: {STANDARD_P_VALUES}")
print(f"  Weights: {STANDARD_P_WEIGHTS}")
print()
for name, d, n in FLAT_DATASETS_TO_GENERATE:
    print(f"  • {name}: d={d}, n={n:,}")

Flat datasets configured for generation:
  Error rates: [0.001, 0.003, 0.005, 0.007]
  Weights: [0.25, 0.25, 0.25, 0.25]

  • d3_baseline: d=3, n=1,000,000
  • d5_baseline: d=5, n=1,000,000
  • d7_baseline: d=7, n=1,000,000
  • d9_baseline: d=9, n=1,000,000


In [3]:
# ============================================================
# GENERATE FLAT DATASETS
# ============================================================
# This cell will generate all configured flat datasets.
# Skip any that already exist.

flat_datasets_dir = BASE_PATH / "datasets" / "flat"

for name, d, n_samples in FLAT_DATASETS_TO_GENERATE:
    # Check if dataset already exists
    if (flat_datasets_dir / f"{name}.pt").exists():
        print(f"⏭️  Flat dataset '{name}' already exists, skipping...")
        continue

    print(f"\n{'='*60}")
    print(f"Generating flat dataset: {name}")
    print(f"{'='*60}")

    # Create and generate dataset
    cache = FlatDatasetCache(base_path=BASE_PATH, device=device)
    cache.generate(
        d=d,
        n_samples=n_samples,
        p_values=STANDARD_P_VALUES,
        p_weights=STANDARD_P_WEIGHTS,
        verbose=True
    )

    # Save to disk
    cache.save(name)

    print(f"✅ Flat dataset '{name}' saved successfully!")

    # Cleanup
    del cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("Flat dataset generation complete!")
print(f"{'='*60}")

⏭️  Flat dataset 'd3_baseline' already exists, skipping...
⏭️  Flat dataset 'd5_baseline' already exists, skipping...
⏭️  Flat dataset 'd7_baseline' already exists, skipping...
⏭️  Flat dataset 'd9_baseline' already exists, skipping...

Flat dataset generation complete!


#### List existing flat datasets

In [4]:
# List all cached flat datasets
datasets = FlatDatasetCache.list_datasets(base_path=BASE_PATH)

if not datasets:
    print("No cached flat datasets found.")
    print("Run the generation cells above to create some!")
else:
    print(f"Found {len(datasets)} cached flat dataset(s):\n")
    for ds in datasets:
        print(f"📁 {ds['name']}")
        print(f"   Samples: {ds.get('n_samples', 'unknown'):,}")
        print(f"   Distance: d={ds.get('d', '?')}")
        print(f"   Detectors: {ds.get('num_detectors', '?')}")
        print(f"   Error rates: {ds.get('p_values', '?')}")
        print(f"   Size: {ds.get('size_mb', 0):.1f} MB")
        print()

Found 4 cached flat dataset(s):

📁 d3_baseline
   Samples: 1,000,000
   Distance: d=3
   Detectors: 24
   Error rates: [0.001, 0.003, 0.005, 0.007]
   Size: 95.4 MB

📁 d5_baseline
   Samples: 1,000,000
   Distance: d=5
   Detectors: 120
   Error rates: [0.001, 0.003, 0.005, 0.007]
   Size: 461.6 MB

📁 d7_baseline
   Samples: 1,000,000
   Distance: d=7
   Detectors: 336
   Error rates: [0.001, 0.003, 0.005, 0.007]
   Size: 1285.6 MB

📁 d9_baseline
   Samples: 1,000,000
   Distance: d=9
   Detectors: 720
   Error rates: [0.001, 0.003, 0.005, 0.007]
   Size: 2750.4 MB



#### Training

#### Definition

In [5]:
class SimpleNNTrainer:
    """
    Highly customizable trainer for SimpleNN models on quantum error correction data.

    This trainer handles the entire training pipeline:
    - Loads pre-generated flat arrays from cache OR generates them on-the-fly
    - Trains the SimpleNN model
    - Returns the trained model

    Supports three modes:
    1. Pre-loaded data: Pass detections/labels directly
    2. Cached datasets: Load from disk via `cache_name` parameter
    3. On-the-fly generation: Generate data during training

    Attributes:
        nickname (str): Human-readable name for the model
        d (int): Code distance
        p_values (list[float]): Physical error rates
        p_weights (list[float]): Distribution of error rates
        sample_size (int): Total number of training samples
        epochs (int): Number of training epochs
        batch_size (int): Training batch size
        lr (float): Learning rate
        hidden_dim (int): Hidden dimension for the network
        device (torch.device): Compute device
    """

    def __init__(
        self,
        nickname: str,
        d: int = None,
        p_values: list[float] = None,
        p_weights: list[float] = None,
        sample_size: int = None,
        epochs: int = 10,
        batch_size: int = 200,
        lr: float = 1e-3,
        hidden_dim: int = 256,
        device: torch.device = None,
        base_path: Path = None,
        cache_name: str = None,
        seed: int = None
    ):
        """
        Initialize the SimpleNN trainer.

        Args:
            nickname: Human-readable name for the model
            d: Code distance (not needed if using cache)
            p_values: Physical error rates (not needed if using cache)
            p_weights: Distribution of error rates (not needed if using cache)
            sample_size: Total samples to generate (not needed if using cache)
            epochs: Number of training epochs (default: 10)
            batch_size: Training batch size (default: 200)
            lr: Learning rate (default: 1e-3)
            hidden_dim: Hidden dimension (default: 256)
            device: Compute device (defaults to CUDA if available)
            base_path: Base path for model/dataset storage
            cache_name: Name of cached dataset to load (optional)
            seed: Random seed for reproducibility
        """
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.base_path = base_path
        self._cache_name = cache_name

        # Determine data source mode
        if cache_name is not None:
            self._mode = "cached"
            # Load cache to get config
            cache = FlatDatasetCache(base_path=base_path, device=self.device)
            cache.load(cache_name, verbose=False)
            self._detections = cache.detections
            self._labels = cache.labels
            sample_size = len(cache)
            d = cache.config.get('d')
            p_values = cache.config.get('p_values')
            p_weights = cache.config.get('p_weights')
        else:
            self._mode = "generate"
            self._detections = None
            self._labels = None

            if d is None or p_values is None or sample_size is None:
                raise ValueError("For on-the-fly generation, must provide d, p_values, and sample_size")

            if p_weights is None:
                p_weights = [1.0 / len(p_values)] * len(p_values)

        # Store configuration
        self.nickname = nickname
        self.d = d
        self.p_values = p_values
        self.p_weights = p_weights
        self.sample_size = sample_size
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.hidden_dim = hidden_dim
        self.seed = seed

        print(f"SimpleNNTrainer initialized:")
        print(f"  Nickname: {nickname}")
        print(f"  Mode: {self._mode}")
        if self._mode == "cached":
            print(f"  Cache: {cache_name}")
        print(f"  Distance: d={d}")
        print(f"  Error rates: {p_values}")
        print(f"  Sample size: {sample_size:,}")
        print(f"  Epochs: {epochs}, Batch size: {batch_size}, LR: {lr}")
        print(f"  Hidden dim: {hidden_dim}")
        print(f"  Device: {self.device}")

    def train(self, verbose: bool = True) -> SimpleNN:
        """
        Execute the full training pipeline.

        Args:
            verbose: Print progress information (default: True)

        Returns:
            SimpleNN: The trained SimpleNN model instance
        """
        import random

        if verbose:
            print(f"\n{'='*60}")
            print(f"Starting training pipeline for: {self.nickname}")
            print(f"{'='*60}")

        # Get data based on mode
        if self._mode == "cached":
            detections = self._detections
            labels = self._labels
            if verbose:
                print(f"\nUsing cached data: {len(labels):,} samples")
        else:
            # Generate data on-the-fly
            if verbose:
                print(f"\nGenerating data on-the-fly...")
            sampler = SurfaceCodeSampler(p=self.p_values[0], device=self.device)
            detections, labels = sampler.sample(
                d=self.d,
                num_samples=self.sample_size,
                p_values=self.p_values,
                p_weights=self.p_weights
            )

        # Shuffle data
        if verbose:
            print(f"Shuffling {len(labels):,} samples...")
        perm = torch.randperm(len(labels), device=self.device)
        detections = detections[perm]
        labels = labels[perm]

        # Get number of detectors
        num_detectors = detections.shape[1]

        # Initialize model
        if verbose:
            print(f"\nInitializing SimpleNN model...")
            print(f"  Input features: {num_detectors}")

        model = SimpleNN(
            nickname=self.nickname,
            in_channels=num_detectors,
            hidden_dim=self.hidden_dim,
            device=self.device,
            base_path=self.base_path,
            seed=self.seed
        )

        # Training setup
        model.model.train()
        optimizer = torch.optim.Adam(model.model.parameters(), lr=self.lr)
        loss_fn = torch.nn.BCELoss()

        num_batches = len(labels) // self.batch_size
        total_batches = num_batches * self.epochs

        if verbose:
            print(f"\nStarting training...")
            print(f"  Total batches: {total_batches:,}")

        epoch_losses = []
        pbar = tqdm(total=total_batches, desc="Training", disable=not verbose)

        for epoch in range(self.epochs):
            epoch_loss = 0.0
            running_acc = 0.0

            for batch_idx in range(num_batches):
                start_idx = batch_idx * self.batch_size
                end_idx = start_idx + self.batch_size

                X = detections[start_idx:end_idx]
                y = labels[start_idx:end_idx].unsqueeze(1)

                # Forward pass
                pred = model.model(X)
                loss = loss_fn(pred, y)

                # Accuracy
                acc = ((pred > 0.5).float() == y).float().mean().item()
                running_acc = acc * 0.1 + running_acc * 0.9 if running_acc else acc

                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                pbar.update(1)

                if batch_idx % 100 == 0:
                    pbar.set_postfix({
                        'epoch': f'{epoch+1}/{self.epochs}',
                        'loss': f'{loss.item():.4f}',
                        'acc': f'{running_acc:.4f}'
                    })

            avg_loss = epoch_loss / num_batches
            epoch_losses.append(avg_loss)

        pbar.close()

        if verbose:
            print(f"\n{'='*60}")
            print(f"Training complete for: {self.nickname}")
            print(f"Final loss: {epoch_losses[-1]:.4f}")
            print(f"{'='*60}")

        model.model.eval()
        return model

    def __repr__(self) -> str:
        return (f"SimpleNNTrainer(nickname='{self.nickname}', "
                f"d={self.d}, "
                f"sample_size={self.sample_size:,}, "
                f"epochs={self.epochs})")

#### Logic

Train SimpleNN models across multiple distances and seeds (like GraphSAGE baseline training).

In [6]:
# Directory where models are saved
models_dir = BASE_PATH / "models" / "simple_nn"
models_dir.mkdir(parents=True, exist_ok=True)

# Distances to train (using cached flat datasets)
distances = [3, 5, 7, 9]
seeds = [1, 2, 3, 4, 5]

for d in distances:
    for seed in seeds:
        nickname = f"d{d}_baseline_seed_{seed}"
        cache_name = f"d{d}_baseline"

        # Check if model already exists
        existing = list(models_dir.glob(f"{nickname}_*.pt"))
        if existing:
            print(f"Model '{nickname}' already exists: {existing[0].name}, skipping...")
            continue

        # Check if flat dataset exists
        flat_data_path = BASE_PATH / "datasets" / "flat" / f"{cache_name}.pt"
        if not flat_data_path.exists():
            print(f"Flat dataset '{cache_name}' not found, skipping d={d}...")
            print(f"  Run the dataset generation cell first!")
            continue

        print(f"\n{'='*60}")
        print(f"Training {nickname}...")
        print(f"{'='*60}")

        trainer = SimpleNNTrainer(
            nickname=nickname,
            cache_name=cache_name,  # Use cached flat dataset
            epochs=10,
            batch_size=256,
            lr=1e-3,
            hidden_dim=256,
            base_path=BASE_PATH,
            device=device,
            seed=seed
        )

        # Train and save immediately
        model = trainer.train()
        model.save(nickname)
        print(f"Saved model: {nickname}")

        # Clean up to free RAM
        del model
        del trainer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"Memory cleared.")

print(f"\nDone! Training complete.")

Model 'd3_baseline_seed_1' already exists: d3_baseline_seed_1_2026-01-14_17-42-11.pt, skipping...
Model 'd3_baseline_seed_2' already exists: d3_baseline_seed_2_2026-01-14_17-45-19.pt, skipping...

Training d3_baseline_seed_3...
SimpleNNTrainer initialized:
  Nickname: d3_baseline_seed_3
  Mode: cached
  Cache: d3_baseline
  Distance: d=3
  Error rates: [0.001, 0.003, 0.005, 0.007]
  Sample size: 1,000,000
  Epochs: 10, Batch size: 256, LR: 0.001
  Hidden dim: 256
  Device: cuda

Starting training pipeline for: d3_baseline_seed_3

Using cached data: 1,000,000 samples
Shuffling 1,000,000 samples...

Initializing SimpleNN model...
  Input features: 24
SimpleNN initialized: SimpleNN(nickname='d3_baseline_seed_3', in_channels=24, hidden_dim=256)

Starting training...
  Total batches: 39,060


Training: 100%|██████████| 39060/39060 [03:20<00:00, 195.04it/s, epoch=10/10, loss=0.0287, acc=0.9864]



Training complete for: d3_baseline_seed_3
Final loss: 0.0359
Model saved to: ..\models\simple_nn\d3_baseline_seed_3_2026-01-14_18-00-51.pt
Saved model: d3_baseline_seed_3
Memory cleared.

Training d3_baseline_seed_4...
SimpleNNTrainer initialized:
  Nickname: d3_baseline_seed_4
  Mode: cached
  Cache: d3_baseline
  Distance: d=3
  Error rates: [0.001, 0.003, 0.005, 0.007]
  Sample size: 1,000,000
  Epochs: 10, Batch size: 256, LR: 0.001
  Hidden dim: 256
  Device: cuda

Starting training pipeline for: d3_baseline_seed_4

Using cached data: 1,000,000 samples
Shuffling 1,000,000 samples...

Initializing SimpleNN model...
  Input features: 24
SimpleNN initialized: SimpleNN(nickname='d3_baseline_seed_4', in_channels=24, hidden_dim=256)

Starting training...
  Total batches: 39,060


Training:  40%|████      | 15631/39060 [01:39<03:36, 108.05it/s, epoch=5/10, loss=0.0454, acc=0.9883]

KeyboardInterrupt: 

Training:  40%|████      | 15633/39060 [01:49<03:36, 108.05it/s, epoch=5/10, loss=0.0454, acc=0.9883]